# C for A: Customer Relationship EDA

My slice of Workstream A: does the **shape of the relationship** predict churn?
Tenure, contract type, payment method. Demographics, products and churn reasons are A's.

1. Setup and first look at the data
2. Tenure: when do customers leave?
3. Contract, and does it survive controlling for tenure?
4. Payment method, or is that just contract again?
5. Risk profile: stacking the factors
6. Revenue at risk
7. Hand-offs

Everything is measured as a **churn rate** against the portfolio average, always with `n`.
Not accuracy. Guessing "nobody churns" would be 73.5% accurate and catch no one.

## 1 · Setup and first look

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

Path("figs").mkdir(exist_ok=True)

# Swan brand palette. Navy is sampled from the logo wordmark; teal, coral and grey
# stay as the agreed data colours. Charts sit on the same paper as the slides so
# they drop into Canva without a white box around them.
navy, slate, paper, rule = "#163e67", "#55697c", "#f7f9fb", "#dce3ea"
teal, coral, grey = "#136e78", "#d1495b", "#9aa5a8"

plt.rcParams.update({
    "font.size": 11,
    "savefig.dpi": 200,                 # 2x for the deck
    "savefig.bbox": "tight",
    "figure.facecolor": paper,
    "axes.facecolor": paper,
    "savefig.facecolor": paper,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": rule,
    "axes.labelcolor": slate,
    "xtick.color": slate,
    "ytick.color": navy,
    "text.color": navy,
    "axes.grid": True,
    "grid.color": rule,
    "axes.axisbelow": True,
})

In [ ]:
df = pd.read_excel("../Swan Consulting 1 - Project Data.xlsx", sheet_name="Telco_Churn")
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

`Total Charges` is an object column, which is odd for a money field. Worth a look.

In [ ]:
df["Total Charges"].head(10)

In [ ]:
# how many are not numbers?
pd.to_numeric(df["Total Charges"], errors="coerce").isna().sum()

In [ ]:
blank = pd.to_numeric(df["Total Charges"], errors="coerce").isna()
df.loc[blank, ["CustomerID", "Tenure Months", "Monthly Charges", "Churn Value"]]

All 11 have **zero tenure**. They are brand new customers who haven't been billed yet.
So the true value is 0, not missing. None of them have churned.

In [ ]:
df.isna().sum()

`Churn Reason` has 5,174 nulls. That's exactly the number of customers who *stayed*,
so it's only filled in for churners. Useful for A's reporting, but it would leak the
answer straight into a model.

In [ ]:
print(df["Churn Reason"].notna().sum(), "customers have a churn reason")
print(df["Churn Value"].sum(), "customers churned")
df["Churn Reason"].value_counts().head()

In [ ]:
df.nunique().sort_values().head(8)

`Count`, `Country` and `State` have one value each, so they carry no information.
`Churn Label` is just the text version of `Churn Value`. `City` has 1,129 values,
fine for reporting but far too many to one-hot encode.

### Cleaning

| Column | Issue | Fix |
|---|---|---|
| `Total Charges` | text, 11 blanks at zero tenure | to numeric, fill 0 |
| `Count`, `Country`, `State` | one value each | drop |
| `Churn Label` | duplicate of `Churn Value` | drop |
| `Churn Reason` | churners only | keep for reporting, never model |
| `No internet service` | a real state, not missing | leave as is |

Team note: A and B use this same cell so we all clean identically.

In [ ]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)
df = df.drop(columns=["Count", "Country", "State", "Churn Label"])

df["tenure_band"] = pd.cut(df["Tenure Months"], [-1, 6, 12, 24, 48, 72],
                           labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
df["is_autopay"] = df["Payment Method"].str.contains("automatic").astype(int)
df["is_echeck"] = (df["Payment Method"] == "Electronic check").astype(int)
df["is_paperless"] = (df["Paperless Billing"] == "Yes").astype(int)
df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype(int)

df.shape

In [ ]:
df.describe()

In [ ]:
N = len(df)
churners = df["Churn Value"].sum()
BASE = df["Churn Value"].mean()

print(f"customers  {N:,}")
print(f"churned    {churners:,}")
print(f"baseline   {BASE:.1%}")

26.5% is the number every other number gets compared to. Two small helpers so the
charts stay consistent across all three notebooks.

In [ ]:
def churn_rate(col):
    t = df.groupby(col, observed=True)["Churn Value"].agg(n="size", churn_rate="mean")
    return t.sort_values("churn_rate", ascending=False)


def churn_bar(table, title, subtitle="", figsize=None):
    t = table.sort_values("churn_rate")
    colours = [coral if v >= BASE else teal for v in t["churn_rate"]]

    fig, ax = plt.subplots(figsize=figsize or (9, 0.6 * len(t) + 2))
    ax.barh(t.index.astype(str), t["churn_rate"], color=colours, height=0.65)
    ax.axvline(BASE, ls="--", color=slate, lw=1.4)

    for i, (rate, n) in enumerate(zip(t["churn_rate"], t["n"])):
        ax.text(rate + 0.01, i, f"{rate:.1%}   n={n:,}", va="center", fontsize=10)

    ax.text(BASE - 0.004, 0.99, f"average {BASE:.1%}", rotation=90, fontsize=9,
            color=slate, ha="right", va="top",
            transform=ax.get_xaxis_transform(), bbox=dict(fc=paper, ec="none"))

    ax.set_xlim(0, t["churn_rate"].max() * 1.3)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.set_xlabel("Churn rate")
    ax.grid(axis="y", visible=False)
    ax.set_title(title, loc="left", fontsize=13.5, fontweight="bold", pad=24)
    ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10, color=slate)
    fig.tight_layout()
    return fig, ax

## 2 · Tenure

If churn is spread evenly across a customer's life you need a broad retention programme.
If it's concentrated early, you need an onboarding fix. So: when do people actually leave?

In [ ]:
df.groupby("Churn Value")["Tenure Months"].describe()

Churners average 18 months against 37.6 for stayers, and the medians are further
apart still, 10 months vs 38.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.4))

ax.hist([df.loc[df["Churn Value"] == 0, "Tenure Months"],
         df.loc[df["Churn Value"] == 1, "Tenure Months"]],
        bins=np.arange(0, 74, 3), stacked=True,
        color=[teal, coral], label=["Stayed", "Churned"])

ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Customers")
ax.set_ylim(0, 980)
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
ax.set_title("More than half of all churn happens in the first 12 months",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, "Customer count by tenure, split by outcome  |  n=7,043",
        transform=ax.transAxes, fontsize=10, color=slate)

ax.annotate("", xy=(0.3, 900), xytext=(12, 900),
            arrowprops=dict(arrowstyle="<->", color=slate))
ax.text(13.5, 900, "1,037 churners = 55% of all churn", va="center",
        fontsize=9.5, color=slate)

fig.tight_layout()
fig.savefig("figs/tenure_distribution.png")

The coral block collapses after month 12. The spike at 70+ months is long-standing
customers who almost never leave.

Same thing as a rate, using the bands the team agreed:

In [ ]:
tenure_table = churn_rate("tenure_band")
tenure_table

In [ ]:
fig, ax = churn_bar(tenure_table,
    "Churn rate falls from 53% to 10% as customers pass their first year",
    "Churn rate by tenure band (months)  |  n=7,043")
fig.savefig("figs/churn_by_tenure_band.png")

In [ ]:
first_year = df[df["Tenure Months"] <= 12]
first_six = df[df["Tenure Months"] <= 6]

print(f"under 12 months: {len(first_year):,} customers ({len(first_year)/N:.0%} of base)")
print(f"  they are {first_year['Churn Value'].sum()/churners:.1%} of all churn")
print(f"under 6 months : {first_six['Churn Value'].sum()/churners:.1%} of all churn")

> **Finding 1: churn is front-loaded.** Under-one-year customers are 31% of the base
> but **55% of all churn**. The 0-6 month band churns at **52.9%**, twice the portfolio
> average. Median churner leaves at 10 months; median stayer is 38 months old.
>
> This is an onboarding problem before it is a loyalty problem.

## 3 · Contract

The obvious next cut. But contract and tenure are tangled together. A two-year customer
has had to survive two years to still be on the books. So look at the headline first,
then hold tenure constant and see if it holds up.

In [ ]:
df["Contract"].value_counts()

In [ ]:
pd.crosstab(df["Contract"], df["Churn Value"])

In [ ]:
contract_table = churn_rate("Contract")
contract_table

In [ ]:
fig, ax = churn_bar(contract_table,
    "Month-to-month customers churn at 15x the two-year rate",
    "Churn rate by contract type  |  n=7,043")
fig.savefig("figs/churn_by_contract.png")

In [ ]:
r = contract_table["churn_rate"]
print(f"month-to-month vs two-year: {r['Month-to-month'] / r['Two year']:.1f}x")

share = df[df["Churn Value"] == 1]["Contract"].value_counts(normalize=True)
print(f"\nmonth-to-month is {df['is_month_to_month'].mean():.0%} of the base")
print(f"but {share['Month-to-month']:.0%} of all churners")

### Does it survive controlling for tenure?

Same comparison, but inside each tenure band. If contract is really just tenure in
disguise, the gap should close as customers get older.

In [ ]:
rate = df.pivot_table(index="tenure_band", columns="Contract",
                      values="Churn Value", aggfunc="mean", observed=True)
count = df.pivot_table(index="tenure_band", columns="Contract",
                       values="Churn Value", aggfunc="size", observed=True)

order = ["Month-to-month", "One year", "Two year"]
rate, count = rate[order], count[order]
(rate * 100).round(1)

In [ ]:
count

In [ ]:
bands = list(rate.index)[::-1]
y = np.arange(len(bands))
h = 0.25

fig, ax = plt.subplots(figsize=(10.5, 5.6))

for i, (contract, colour) in enumerate(zip(order, [coral, grey, teal])):
    vals = rate.loc[bands, contract]
    ns = count.loc[bands, contract]
    ax.barh(y + (1 - i) * h, vals, height=h * 0.9, color=colour, label=contract)
    for yy, v, n in zip(y + (1 - i) * h, vals, ns):
        ax.text(v + 0.012, yy, f"{v:.0%}  n={n:,}", va="center", fontsize=9)
        if v == 0:
            ax.plot(0, yy, marker="|", ms=9, mew=2, color=colour)

ax.axvline(BASE, ls="--", color=slate, lw=1.4)
ax.text(BASE - 0.004, 0.99, f"average {BASE:.1%}", rotation=90, fontsize=9,
        color=slate, ha="right", va="top",
        transform=ax.get_xaxis_transform(), bbox=dict(fc=paper, ec="none"))

ax.set_yticks(y)
ax.set_yticklabels(bands)
ax.set_xlim(0, 0.72)
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.set_xlabel("Churn rate")
ax.set_ylabel("Tenure band (months)")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Contract beats tenure: even 4+ year month-to-month customers churn at 26%",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, "Churn rate by contract, within each tenure band",
        transform=ax.transAxes, fontsize=10, color=slate)

fig.tight_layout()
fig.savefig("figs/contract_within_tenure.png")

In [ ]:
print("month-to-month churn by tenure band:")
for b in rate.index:
    print(f"  {b:>6}  {rate.loc[b, 'Month-to-month']:.1%}   n={count.loc[b, 'Month-to-month']:,}")

mtm = rate.loc["49-72", "Month-to-month"]
two = rate.loc["49-72", "Two year"]
print(f"\nlongest-tenured band: {mtm:.1%} vs {two:.1%} = {mtm/two:.1f}x at the same tenure")

> **Finding 2: contract is not tenure in disguise.** Month-to-month churns at
> **42.7%** against **2.8%** on two-year. The gap survives the tenure control: among
> customers of 49-72 months, month-to-month still churns at **26.0%** while two-year
> customers of the same age churn at **3.3%**, a **7.8x** gap. Two-year customers stay
> under 4% in every band.
>
> The month-to-month rate does fall with tenure (55% → 26%), but never reaches the
> portfolio average.
>
> *Caveat:* the two-year 0% cells below 25 months (n=29/39/90) are customers still inside
> their initial term, so that zero is structural, not good behaviour.
>
> Contract migration is the highest-leverage retention lever we have.

## 4 · Payment method

Contract is hard to change, because you have to persuade someone to sign for two years.
How they pay is much easier to change, so if it carries signal it's a cheaper lever.

In [ ]:
df["Payment Method"].value_counts()

In [ ]:
payment_table = churn_rate("Payment Method")
payment_table

In [ ]:
fig, ax = churn_bar(payment_table,
    "Electronic-check payers churn at 45%, triple the credit-card rate",
    "Churn rate by payment method  |  n=7,043", figsize=(9.8, 4.4))
fig.savefig("figs/churn_by_payment.png")

The two automatic methods sit together at the bottom and the two manual ones at the
top, so the split that matters is really automatic vs manual. Same for paperless billing:

In [ ]:
billing = pd.concat([
    churn_rate("is_autopay").rename(index={1: "On autopay", 0: "Manual payment"}),
    churn_rate("Paperless Billing").rename(index={"Yes": "Paperless billing",
                                                  "No": "Paper billing"}),
])
billing

In [ ]:
fig, ax = churn_bar(billing,
    "Customers on autopay churn at half the rate of manual payers",
    "Churn rate by billing setup  |  n=7,043", figsize=(9.8, 4.4))
fig.savefig("figs/autopay_paperless.png")

### Is e-check just month-to-month again?

Most e-check users are on month-to-month contracts, so the two are confounded.
Check how badly:

In [ ]:
pd.crosstab(df["Payment Method"], df["Contract"], normalize="index").round(2)

78% of e-check users are month-to-month, so some of that 45% is really the contract
showing through. Hold contract constant and see what's left:

In [ ]:
pd.crosstab(df["Contract"], df["is_echeck"], values=df["Churn Value"],
            aggfunc="mean").round(3)

In [ ]:
# and holding tenure constant as well
for label, rows in [("month-to-month, under 12 months",
                     df[(df["is_month_to_month"] == 1) & (df["Tenure Months"] < 12)]),
                    ("month-to-month, 24+ months",
                     df[(df["is_month_to_month"] == 1) & (df["Tenure Months"] >= 24)])]:
    g = rows.groupby("is_echeck")["Churn Value"].agg(["size", "mean"])
    gap = (g.loc[1, "mean"] - g.loc[0, "mean"]) * 100
    print(f"{label}")
    print(f"  e-check {g.loc[1,'mean']:.1%} (n={g.loc[1,'size']:,})"
          f"  vs other {g.loc[0,'mean']:.1%} (n={g.loc[0,'size']:,})  -> +{gap:.0f}pp\n")

> **Finding 3: payment method carries its own signal.** Electronic check churns at
> **45.3%** against 15.2% on credit card. It is confounded with contract (78% of e-check
> users are month-to-month), but it is not *only* that: holding contract **and** tenure
> constant, e-check still adds **+23pp** among month-to-month customers under a year and
> **+15pp** among those past two years.
>
> Collapsed to the useful flag: manual payers churn at **34.7%**, autopay customers at
> **16.0%**. Paperless billing points the same way (33.6% vs 16.3%).
>
> Moving customers onto autopay is cheap, fast, and needs no new contract.

## 5 · The risk profile

Three fields, all visible the moment a customer calls in. What does the worst combination
look like, and how many customers are in it?

In [ ]:
df["risky"] = ((df["is_month_to_month"] == 1) &
               (df["is_echeck"] == 1) &
               (df["Tenure Months"] < 12)).astype(int)

df["safest"] = ((df["Contract"] == "Two year") &
                (df["is_autopay"] == 1) &
                (df["Tenure Months"] >= 24)).astype(int)

steps = {
    "Whole customer base": df["Churn Value"].notna(),
    "Month-to-month": df["is_month_to_month"] == 1,
    "…paying by e-check": (df["is_month_to_month"] == 1) & (df["is_echeck"] == 1),
    "…and under 12 months old": df["risky"] == 1,
    "Safest profile:\n2-year + autopay + 2 yrs tenure": df["safest"] == 1,
}

ladder = pd.DataFrame([
    {"step": name, "n": mask.sum(), "churn_rate": df.loc[mask, "Churn Value"].mean()}
    for name, mask in steps.items()
]).set_index("step")
ladder

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

labels = list(ladder.index)[::-1]
vals = list(ladder["churn_rate"])[::-1]
ns = list(ladder["n"])[::-1]
colours = [teal, coral, coral, coral, grey]

ax.barh(labels, vals, color=colours, height=0.65)
ax.axvline(BASE, ls="--", color=slate, lw=1.4)
ax.text(BASE - 0.004, 0.99, f"average {BASE:.1%}", rotation=90, fontsize=9,
        color=slate, ha="right", va="top",
        transform=ax.get_xaxis_transform(), bbox=dict(fc=paper, ec="none"))

for i, (v, n) in enumerate(zip(vals, ns)):
    ax.text(v + 0.012, i, f"{v:.1%}   n={n:,}  ({n/N:.0%} of base)", va="center",
            fontsize=10.5, bbox=dict(fc=paper, ec="none"))

ax.set_xlim(0, 0.92)
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.set_xlabel("Churn rate")
ax.grid(axis="y", visible=False)
ax.set_title("Three fields find a 13% slice of the base that loses two customers in three",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, "Stacking the risk factors  |  riskiest profile churns 22x the safest",
        transform=ax.transAxes, fontsize=10, color=slate)

fig.tight_layout()
fig.savefig("figs/risk_ladder.png")

In [ ]:
risky = df[df["risky"] == 1]
safest = df[df["safest"] == 1]

print(f"riskiest: {len(risky):,} customers, {risky['Churn Value'].mean():.1%} churn")
print(f"  that is {len(risky)/N:.0%} of the base holding "
      f"{risky['Churn Value'].sum()/churners:.0%} of all churn")
print(f"safest  : {len(safest):,} customers, {safest['Churn Value'].mean():.1%} churn")
print(f"\nthe risky group is {len(risky)/500:.1f}x the 500-mailer budget")

> **Finding 4: three columns and no model get you a long way.** Month-to-month
> + e-check + under 12 months = **917 customers (13% of the base) churning at 63.9%**,
> holding **31% of all churn**. The mirror-image profile churns at **2.9%**, a **22x**
> spread.
>
> But 917 is 1.8x the 500-mailer budget, so a rule alone can't pick the list. It finds the
> pool; B's ranked probabilities have to order it.

### Why this still needs a model

Everything above is a **group rate** measured on customers whose outcome we already know:
"of these 917 people, 586 left". The deliverables need something different: a number
*per customer*, for people whose outcome we don't know yet.

Three reasons the manual version can't get there:

- **Ties.** All 917 risky customers score the same 63.9%. To send 500 mailers we'd be
  picking 500 of 917 at random. A model gives each of them a different probability, so
  they can be ranked and cut at 500.
- **Empty cells.** Three fields give 8 combinations, which is fine. The 19 usable fields
  give millions, and almost all of them contain nobody. A rate off n=2 is noise.
- **Borrowing strength.** A model estimates the effect of e-check from *all* 2,365 e-check
  customers at once, so it can still score a combination it has never seen.

So the split is: this notebook explains **why** people churn and hands B the features;
B's model says **who**, with a probability for all 7,043.

It also leaves B a benchmark to beat. Any model worth using should pick 500 customers
with better than 63.9% precision, since that rule already does.

## 6 · Revenue at risk

Rates alone don't set priorities. A high churn rate on cheap customers may cost less than
a low one on expensive customers. So attach money to it.

In [ ]:
df.groupby("Churn Value")["Monthly Charges"].mean().round(2)

Churners pay **more** than customers who stay, so this isn't bargain-hunters leaving.
It also means the brief's `n × rate × average charges` shorthand understates the loss,
because the churners in a segment are above its average. Both columns below; I use the actual one.

In [ ]:
lost = df[df["Churn Value"] == 1]
total_lost = lost["Monthly Charges"].sum()

revenue = pd.DataFrame({
    "n": df.groupby("Contract")["Churn Value"].size(),
    "churn_rate": df.groupby("Contract")["Churn Value"].mean(),
    "churners": lost.groupby("Contract").size(),
    "lost_monthly": lost.groupby("Contract")["Monthly Charges"].sum(),
})
revenue["lost_yearly"] = revenue["lost_monthly"] * 12
revenue["share"] = revenue["lost_monthly"] / total_lost
revenue.round(3)

In [ ]:
print(f"total lost: ${total_lost:,.0f} per month = ${total_lost*12:,.0f} per year")

for name, mask in [("riskiest profile", df["risky"] == 1),
                   ("electronic check", df["is_echeck"] == 1),
                   ("first 12 months", df["Tenure Months"] <= 12)]:
    seg = df[mask & (df["Churn Value"] == 1)]["Monthly Charges"].sum()
    print(f"{name:18s} ${seg*12:,.0f} per year  ({seg/total_lost:.0%} of the loss)")

In [ ]:
fig, ax = plt.subplots(figsize=(9.6, 3.9))

r = revenue.sort_values("lost_yearly")
ax.barh(r.index, r["lost_yearly"], color=[teal, grey, coral], height=0.6)

for i, (v, c) in enumerate(zip(r["lost_yearly"], r["churners"])):
    ax.text(v + 30000, i, f"${v/1e6:.2f}m   {c:,} churners", va="center", fontsize=10.5)

ax.set_xlim(0, 2_000_000)
ax.set_xticks([0, 500_000, 1_000_000, 1_500_000])
ax.set_xticklabels(["$0.0m", "$0.5m", "$1.0m", "$1.5m"])
ax.set_xlabel("Annual recurring revenue lost")
ax.grid(axis="y", visible=False)
ax.set_title("Month-to-month churn costs $1.45m a year, 87% of all revenue lost",
             loc="left", fontsize=13.5, fontweight="bold", pad=24)
ax.text(0, 1.02, f"Annualised monthly charges of churned customers  |  "
                 f"${total_lost*12/1e6:.2f}m total",
        transform=ax.transAxes, fontsize=10, color=slate)

fig.tight_layout()
fig.savefig("figs/revenue_at_risk.png")

> **Finding 5: churn costs \\$1.67m a year.** Month-to-month accounts for **87%** of it
> ($1.45m) and electronic check for **61%** (\\$1.01m). The 917-customer risky profile alone
> carries **\\$508k a year**.
>
> Churners average **\\$74.44/month against \\$61.27** for stayers. This is high-value
> attrition, which is what justifies spending on the mailer in the first place.

## 7 · Hand-offs

### Summary

Swan loses 1,869 of 7,043 customers, a **26.5% churn rate costing $1.67m a year** in
recurring revenue.

Churn is front-loaded. Customers under a year old are 31% of the base but **55% of all
churn**, and the first six months churn at 52.9%. The median churner leaves at ten months
against 38 for those who stay.

Contract type is the strongest lever, and it isn't simply tenure in disguise.
Month-to-month customers churn at **42.7%** against **2.8%** on two-year deals; holding
tenure constant, month-to-month customers of four years or more still churn at **26.0%**
while two-year customers of identical age churn at **3.3%**.

How customers pay matters independently. Electronic-check payers churn at **45.3%** versus
15.2% on credit card, and e-check still adds 15-23 percentage points within month-to-month
customers of the same tenure. Autopay halves churn, **16.0%** against 34.7%.

Combining three fields (month-to-month, electronic check, under twelve months) isolates
**917 customers (13% of the base) churning at 63.9%**, holding 31% of all churn. The safest
profile churns at 2.9%.

Churners pay more than stayers (\$74.44 vs \$61.27 a month), so this is high-value attrition.

### Charts for the deck

| File | Use |
|---|---|
| `figs/risk_ladder.png` | the headline: three fields, 22x spread |
| `figs/contract_within_tenure.png` | contract survives the tenure control |
| `figs/revenue_at_risk.png` | the money slide |

Supporting: `tenure_distribution`, `churn_by_tenure_band`, `churn_by_contract`,
`churn_by_payment`, `autopay_paperless`.

### For B: features

Worth using: `Tenure Months`, `is_month_to_month`, `is_echeck`, `is_autopay`,
`is_paperless`, a `tenure < 12` flag, and the `month-to-month × e-check` interaction,
which is where the 63.9% segment lives.

Watch out for:

- **Never model `Churn Reason` or `Churn Label`.** Churn Reason is filled in for churners
  only, so it hands over the answer.
- **`Total Charges` is roughly `Monthly Charges × Tenure Months`** (r = 0.9996) and
  correlates 0.83 with tenure. It will destabilise the tenure coefficient in a logistic
  regression. Drop it, or keep it and don't read the coefficients individually.
- **`n_addons` is non-monotone**: 0 add-ons churns at 21%, 1 add-on peaks at 46%, 6
  add-ons at 5%. That happens because the 1,526 no-internet customers sit at 0 and rarely churn.
  Interact it with `has_internet`.
- **Don't one-hot `City`** (1,129 values).
- Use `pd.get_dummies(..., dtype=int)`, because bool dummies get dropped by numeric filters.

### Not covered here
Demographics, products, internet type and churn reasons are A's. Flagging one thing for
them: fibre optic churns at 41.9% vs 19.0% for DSL.

In [ ]:
summary = {
    "baseline_churn": BASE,
    "n_customers": N,
    "n_churners": churners,
    "churn_in_first_12m": first_year["Churn Value"].sum() / churners,
    "churn_rate_0_6m": tenure_table.loc["0-6", "churn_rate"],
    "mtm_churn": contract_table.loc["Month-to-month", "churn_rate"],
    "two_year_churn": contract_table.loc["Two year", "churn_rate"],
    "mtm_churn_oldest": rate.loc["49-72", "Month-to-month"],
    "two_year_churn_oldest": rate.loc["49-72", "Two year"],
    "echeck_churn": payment_table.loc["Electronic check", "churn_rate"],
    "autopay_churn": billing.loc["On autopay", "churn_rate"],
    "manual_churn": billing.loc["Manual payment", "churn_rate"],
    "risky_n": len(risky),
    "risky_churn": risky["Churn Value"].mean(),
    "safest_n": len(safest),
    "safest_churn": safest["Churn Value"].mean(),
    "lost_monthly": total_lost,
    "lost_yearly": total_lost * 12,
}

pd.Series(summary).round(4).to_csv("relationship_eda_findings.csv", header=["value"])
pd.Series(summary).round(4)

---
## 8 · Appendix: can churn reasons be linked to customer features?

Worth testing, because if the riskiest customers left for a distinct reason we could write the
mailer around it. Every churner has a recorded reason, so it is checkable.

Start with reasons that **must** line up with a feature if the data is realistic.

In [ ]:
from scipy.stats import chi2_contingency

churned = df[df["Churn Value"] == 1]

def reason_split(reason, group_col, a, b):
    """% of churners in each group who gave this reason."""
    ga = churned[churned[group_col] == a]
    gb = churned[churned[group_col] == b]
    ra = (ga["Churn Reason"] == reason).mean() * 100
    rb = (gb["Churn Reason"] == reason).mean() * 100
    print(f"{reason}")
    print(f"   {a:>14}: {ra:5.2f}%  (n={len(ga):,})")
    print(f"   {b:>14}: {rb:5.2f}%  (n={len(gb):,})")
    print()

reason_split("Long distance charges", "Phone Service", "Yes", "No")
reason_split("Lack of self-service on Website", "Internet Service", "Fiber optic", "No")
reason_split("Deceased", "Senior Citizen", "Yes", "No")

Those results are not just weak. They are impossible.

Customers **without phone service** cite *long distance charges* at exactly the same rate as
customers who have it. Customers with **no internet at all** complain about the *website*.
*Deceased* is no more common among senior citizens.

Now test everything at once: each of the 20 reasons against 8 customer features.

In [ ]:
features = ["Contract", "Internet Service", "Payment Method", "Senior Citizen",
            "Partner", "Dependents", "Paperless Billing", "Online Security"]

results = []
for reason in churned["Churn Reason"].unique():
    for f in features:
        table = pd.crosstab(churned[f], churned["Churn Reason"] == reason)
        if table.shape[1] < 2 or table.values.min() < 1:
            continue
        results.append({"reason": reason, "feature": f,
                        "p_value": chi2_contingency(table)[1]})

results = pd.DataFrame(results).sort_values("p_value")
n_tests = len(results)
hits = (results["p_value"] < 0.05).sum()

print(f"tests run                     {n_tests}")
print(f"p < 0.05                      {hits}")
print(f"expected by chance alone      {0.05 * n_tests:.1f}")
print(f"surviving Bonferroni          {(results['p_value'] < 0.05 / n_tests).sum()}")
print()
print("strongest associations found:")
results.head(5)

In [ ]:
# Does price sensitivity at least track what people pay?
price = churned["Churn Reason"] == "Price too high"
print(f"cite 'price too high': ${churned.loc[price, 'Monthly Charges'].mean():.2f}/month")
print(f"all other churners   : ${churned.loc[~price, 'Monthly Charges'].mean():.2f}/month")

> **Appendix finding: the reason field carries no customer information.**
>
> 149 tests, 7 significant at p<0.05, and 7.5 expected from pure chance. **None survives
> correction for multiple testing.** Customers citing "price too high" pay \\$75.50 a month
> against \\$74.38 for everyone else.
>
> Combined with reasons that are logically impossible for the customers who gave them, the
> conclusion is that **`Churn Reason` was assigned at random when this dataset was built.**
> It does not correlate with anything because it was never generated from anything.
>
> **What this means for the deck**
>
> - The overall mix is still reportable. "A third of churners cite competitors" is what the
>   data says, and the distribution looks deliberately designed.
> - **Never segment it.** "Month-to-month customers leave for competitor offers" would be a
>   statement about noise.
> - This belongs in the limitations appendix. It is also the honest answer to *why don't you
>   personalise the offer by predicted reason*, because the data cannot support it.
> - The real recommendation: Swan's reason field is a dropdown. The richest churn signal they
>   are not capturing is what customers say in **their own words**: exit surveys, call notes,
>   complaint tickets. That is where text analysis would genuinely pay off.